<a href="https://colab.research.google.com/github/AnanyaAsthana/Machine-Learning/blob/main/kddcup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import urllib.request
import zipfile

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score



In [ ]:
url = "https://sci2s.ugr.es/keel/dataset/data/classification/kddcup-5-fold.zip"
zip_name = "kddcup-5-fold.zip"

urllib.request.urlretrieve(url, zip_name)
print("KDDCup 5-fold dataset downloaded.")


KDDCup 5-fold dataset downloaded.


In [ ]:
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall("kddcup")

print("Dataset extracted.")


Dataset extracted.


In [ ]:
def load_keel_dat(path):
    data = []
    with open(path, "r") as f:
        for line in f:
            if not line.startswith("@") and line.strip():
                data.append([x.strip() for x in line.split(",")])
    return pd.DataFrame(data)



In [ ]:
folds = [
    ("kddcup/kddcup-5-1tra.dat", "kddcup/kddcup-5-1tst.dat"),
    ("kddcup/kddcup-5-2tra.dat", "kddcup/kddcup-5-2tst.dat"),
    ("kddcup/kddcup-5-3tra.dat", "kddcup/kddcup-5-3tst.dat"),
    ("kddcup/kddcup-5-4tra.dat", "kddcup/kddcup-5-4tst.dat"),
    ("kddcup/kddcup-5-5tra.dat", "kddcup/kddcup-5-5tst.dat"),
]



In [ ]:
k_values = [1, 3, 5, 7, 9]
k_accuracy = {k: [] for k in k_values}

for fold_no, (train_path, test_path) in enumerate(folds, 1):
    print(f"\nFold {fold_no}")

    train_df = load_keel_dat(train_path)
    test_df = load_keel_dat(test_path)

    X_train = train_df.iloc[:, :-1]
    y_train = train_df.iloc[:, -1]

    X_test = test_df.iloc[:, :-1]
    y_test = test_df.iloc[:, -1]

    # Identify column types
    categorical_cols = X_train.select_dtypes(include="object").columns
    numerical_cols = X_train.select_dtypes(exclude="object").columns

    # Preprocessing
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numerical_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
        ]
    )

    for k in k_values:
        model = Pipeline([
            ("preprocess", preprocessor),
            ("knn", KNeighborsClassifier(n_neighbors=k))
        ])

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        k_accuracy[k].append(acc)

        print(f"  k = {k} → Accuracy = {acc:.4f}")



Fold 1
